<a href="https://colab.research.google.com/github/sittana-afifi/AIMS-KTT-Hackathon---Challenge-S2.T1.3/blob/main/recommender.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Global setup so we don't reload data every time we search
catalog = pd.read_csv('/catalog.csv')
click_log = pd.read_csv('/click_log.csv')

# 1. Pre-calculate Popularity Weight (The Rank 4 Discovery)
clicks = click_log[click_log['clicked'] == 1].copy()
clicks['weight'] = clicks['rank_shown'].apply(lambda r: 1.2 if r >= 3 else 1.0) # Boost starting at Rank 3
pop_scores = clicks.groupby('sku')['weight'].sum().reset_index()
pop_scores.rename(columns={'weight': 'pop_score'}, inplace=True)

# 2. Tag Rwandan Products
catalog['is_rwanda'] = catalog['origin_district'].apply(lambda x: 1 if x != 'Global' else 0)

# 3. Setup Text Search
catalog['metadata'] = catalog['title'] + " " + catalog['category'] + " " + catalog['description']
tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(catalog['metadata'])

def get_local_recommendations(query, top_n=5):
    """
    Search function that applies Text Similarity,
    Position Bias Correction, and the Local Boost Multiplier.
    """
    # Calculate Text Similarity
    query_vec = tfidf.transform([query])
    sim_scores = cosine_similarity(query_vec, tfidf_matrix).flatten()

    # Create temporary scoring dataframe
    results = catalog.copy()
    results = results.merge(pop_scores, on='sku', how='left').fillna(0)

    # Normalize popularity
    if results['pop_score'].max() > 0:
        results['pop_score'] = results['pop_score'] / results['pop_score'].max()

    results['sim_score'] = sim_scores

    # FINAL SCORING: (Similarity + Popularity) * Local Multiplier
    results['final_score'] = (results['sim_score'] * 0.6) + (results['pop_score'] * 0.4)

    # Apply 20% Rwandan Boost (The Nudge)
    results['final_score'] = results.apply(
        lambda x: x['final_score'] * 1.2 if x['is_rwanda'] == 1 else x['final_score'], axis=1
    )

    return results.sort_values(by='final_score', ascending=False).head(top_n)

# LOGIC NOTE: Results from this function are pushed via API
# to a USSD Gateway for non-smartphone artisan alerts.